# llcAPI_test

## See ledger/llcAPI.py

In [1]:
# Load bookkeeping services
import os, sys
from pathlib import Path
if len([p for p in sys.path if 'Ledger' in p]) == 0:
    sys.path.append(os.path.join(os.getcwd(), 'Ledger'))

## TEST : Load profile 

In [2]:
from ledger.LLC import LLC
import datetime
from IPython.display import display, Markdown

# Load LLC and its profile
top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=True, top=top)

# Summarize LLC
dtReport = datetime.datetime.now().strftime('%Y.%m.%d')
display(Markdown(f"### Profile - {dtReport}"))
display(Markdown(f"- **LLC Name: {llc.objName}**"))
display(Markdown(f"- **Year: {llc.yr}**"))

llc:LLC init load _Profile
LLC llcProfile FN /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/llcProfile_WBGroupLLC.json
Profile loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/llcProfile_WBGroupLLC.json
llc:LLC LLC Init Done


### Profile - 2026.03.14

- **LLC Name: WBGroupLLC**

- **Year: 2025**

## Test: Load Bank csv 

In [3]:
from ledger.llcBank import llcBank

bk = llcBank(llc, debug=True)
bk.fetch()

llcBank llcBank Init Done
llcBank llcBank Init Done
llcBank dwnLdCSV: importBankCSV csvBN: WBGroupLLC_WF_20251231.csv
llcBank CSV Loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/BankStmts/WBGroupLLC_WF_20251231.csv
llcAssets llcAssets Init Done
llc:llcAssets llcAssets Init Done
llcAssets ledgerObject.FN: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/Accts/llcAssets_WBGroupLLC.json
llcAssets ledgerObject.FN: /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/Accts/llcAssets_WBGroupLLC.json
ledgerClassify ledgerClassify Init Done


In [4]:
bk.df

,dt,amt,C2,CheckNo,desc,TransType,Acct,AcctSub,TDesc
0,12/29/2025,-177.00,*,NaN,Cash eWithdrawal in Branch 12/29/2025 13:47 PM...,Exp,Acct.Asset.Purchase,p20251229-RV1,Purchase Rental RV
1,12/29/2025,177.00,*,NaN,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member
2,12/26/2025,-135.80,*,NaN,ALLSTATE IND CO INS PYMT DEC024 00000043853221...,Exp,Acct.Cash.Util,Ins_Home,Pay Monthly Util
3,12/18/2025,-50.00,*,NaN,BILL PAY Water-COMWSC RECURRING 38 ON 12-18,Exp,Acct.Cash.Util,Water,Pay Monthly Util
4,12/10/2025,-129.16,*,NaN,BUSINESS TO BUSINESS ACH Pedernales_Elec ELEC_...,Exp,Acct.Cash.Util,Elec,Pay Monthly Util
5,12/01/2025,1500.00,*,NaN,ZELLE FROM NICOLA ROJAS ON 12/01 REF # BBT3528...,Rev,Acct.Cash.Income,c20251001-1,Rental Income
6,11/18/2025,-50.00,*,NaN,BILL PAY Water-COMWSC RECURRING 38 ON 11-18,Exp,Acct.Cash.Util,Water,Pay Monthly Util
7,11/17/2025,-2.00,*,NaN,PURCHASE AUTHORIZED ON 11/15 HAYS CO TX WIMBER...,Exp,Acct.Cash.Expense,hays,Expense: 11/15 hays co tx wimber fort worth tx...
8,11/17/2025,-30.00,*,NaN,PURCHASE AUTHORIZED ON 11/15 HAYS CO TX WIMBER...,Exp,Acct.Cash.Expense,hays,Expense: 11/15 hays co tx wimber san marcos tx...
9,11/17/2025,-19.47,*,NaN,PURCHASE AUTHORIZED ON 11/15 AMAZON MKTPL*B80W...,Exp,Acct.Cash.Expense,amazon,Expense: 11/15 amazon mktpl*b80w8 amzn.com/bil...


## Test: Tranaction Classification

In [6]:
from ledger.llcAssets import llcAssets           
a = llcAssets(llc)
a.fetch()
r = bk.df.iloc[0]
a._matchBk(r)

s  = bk.df.apply(lambda r : a._matchBk(r), axis=1)
[r for r in s if r is not None]

[('Acct.Asset.Purchase', 'p20251229-RV1', 'Purchase Rental RV'),
 ('Acct.Cash.Investment', 'o20250801_1', 'Initial Investment by member'),
 ('Acct.Asset.Purchase',
  'p20250826-805HMD',
  'Purchase Property: 805 High Mesa'),
 ('Acct.Cash.Investment', 'o20250801_1', 'Open Bank Acct Investment'),
 ('Acct.Cash.Investment', 'o20250801_1', 'Initial Investment by member')]

## Test ledget Iterator

In [15]:
bk.df[bk.df.Acct.str.contains('Investment')]

,dt,amt,C2,CheckNo,desc,TransType,Acct,AcctSub,TDesc
1,12/29/2025,177.0,*,NaN,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member
52,08/20/2025,50.0,*,NaN,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...,Rev,Acct.Cash.Investment,o20250801_1,Open Bank Acct Investment
53,08/20/2025,219000.0,*,NaN,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member


In [11]:
# filter transactions X assets
import pandas as pd
aDF = llc.aObj.df
iDF = pd.merge(aDF, bk.df, 
         left_on=['dt','amt'], 
         right_on=['dt', 'amt'],
         how='inner')
iDF[['acct', 'amt', 'desc_x', 'AcctSub']]
bal = ('Balance_Acct.Cash')
iDF.loc[bal] = [llc.objName, float(iDF.amt.sum()),'LLC']
llcBalInvestment = iDF.amt.iloc[-1]
iDF

,acct,amt,desc_x,AcctSub
0,Acct.Cash.Investment,50.00,Open Bank Acct Investment,o20250801_1
1,Acct.Asset.Purchase,-213936.95,Purchase Property: 805 High Mesa,p20250826-805HMD
2,Acct.Cash.Investment,219000.00,Initial Investment by member,o20250801_1
3,Acct.Asset.Purchase,-177.00,Purchase Rental RV,p20251229-RV1
4,Acct.Cash.Investment,177.00,Initial Investment by member,o20250801_1


In [7]:
#import tests.test_stmtBS as tBS
from tests.test_stmtBS import test_stmtBS, _report, _llc
_report(test_stmtBS())

v0.3 stmtBS test suite — PASS
  checks: 10

  ✓ stmtBS load shape + TOTAL row
      rows=43 cols=['Balance', 'Credit', 'Debit', '_lineNo', '_rowNm', 'acct', 'acctSub', 'acctType'] has_TOTAL=True
  ✓ stmtBS immutability + no save()
      rows_same=True save_raises=True
  ✓ stmtBS.to_DF() + to_json()
      df_ok=True json_ok=True
  ✓ Pipeline AggBy → ViewBy('ByAsset') → SortBy
      rows=28 all_assets=True no_total=True
  ✓ v0.3 stmtBS row count parity vs v0.2 stmtBalanceSheet
      v0.3_rows=43 v0.2_rows=43
  ✓ stmtBS.last_check() shape
      keys=['asset', 'balanced', 'equation_diff', 'equity', 'liability'] balanced=False
  ✓ stmtBS_Tax.nSpaceMap shape + flat load()
      forms=['Form1065', 'Sch_K1', 'Form4562'] Form1065_entries=22
  ✓ stmtBS_View.taxAggregates() keys
      keys=['accum_depr', 'ar', 'buildings', 'cash', 'land', 'mortgage']... missing=[]
  ✓ stmtBS_View view() / stats() / is_balanced()
      all_rows=43 asset_rows=28 balanced=False
  ✓ stmtBS_Reports.load() NotImplement

0

In [18]:
from ledger.stmtProfile import stmtProfile


{('Profile', 'Profile.entity.ein', '_lineNo'): 1,
 ('Profile', 'Profile.entity.ein', '_rowNm'): 'Profile.entity.ein',
 ('Profile', 'Profile.entity.ein', 'acctType'): 'Profile',
 ('Profile', 'Profile.entity.ein', 'acct'): 'entity.ein',
 ('Profile', 'Profile.entity.ein', 'acctSub'): '',
 ('Profile', 'Profile.entity.ein', 'value'): '39-3842347',
 ('Profile', 'Profile.entity.taxnum', '_lineNo'): 2,
 ('Profile', 'Profile.entity.taxnum', '_rowNm'): 'Profile.entity.taxnum',
 ('Profile', 'Profile.entity.taxnum', 'acctType'): 'Profile',
 ('Profile', 'Profile.entity.taxnum', 'acct'): 'entity.taxnum',
 ('Profile', 'Profile.entity.taxnum', 'acctSub'): '',
 ('Profile', 'Profile.entity.taxnum', 'value'): '32101694431',
 ('Profile', 'Profile.entity.entity_name', '_lineNo'): 3,
 ('Profile',
  'Profile.entity.entity_name',
  '_rowNm'): 'Profile.entity.entity_name',
 ('Profile', 'Profile.entity.entity_name', 'acctType'): 'Profile',
 ('Profile', 'Profile.entity.entity_name', 'acct'): 'entity.entity_name'

In [21]:
from ledger.stmtProfile import stmtProfile
from ledger.stmtGL import stmtGL_Tax
from ledger.stmtBS import stmtBS_Tax
from ledger.stmtIS import stmtIS_Tax
llc = _llc()

taxObjDict = dict( glObj = stmtGL_Tax(llc),
                  bsObj = stmtGL_Tax(llc),
                  isObj = stmtGL_Tax(llc),
                  pObj = stmtProfile(llc)
                 )




glNS = glObj.nSpaceMap()
bsNS = bsObj.nSpaceMap()
isNS = isObj.nSpaceMap()
pNS = pObj.nSpaceMap()

glDF_ns = pd.DataFrame(glNS)
bsDF_ns = pd.DataFrame(bsNS)
isDF_ns = pd.DataFrame(isNS)
pDF_ns = pd.DataFrame(pNS)

pd.concat([glDF_ns, bsDF_ns, isDF_ns, pDF_ns])

,Form1065,Sch_K1,Form4562,"(Profile, Profile.entity.ein, _lineNo)","(Profile, Profile.entity.ein, _rowNm)","(Profile, Profile.entity.ein, acctType)","(Profile, Profile.entity.ein, acct)","(Profile, Profile.entity.ein, acctSub)","(Profile, Profile.entity.ein, value)","(Profile, Profile.entity.taxnum, _lineNo)",...,"(Profile, Profile.F1065.tax_year, acctType)","(Profile, Profile.F1065.tax_year, acct)","(Profile, Profile.F1065.tax_year, acctSub)","(Profile, Profile.F1065.tax_year, value)","(Profile, Profile.F1065.chk, _lineNo)","(Profile, Profile.F1065.chk, _rowNm)","(Profile, Profile.F1065.chk, acctType)","(Profile, Profile.F1065.chk, acct)","(Profile, Profile.F1065.chk, acctSub)","(Profile, Profile.F1065.chk, value)"
0,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,17.0
1,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,21.0
2,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,80.0
3,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,86.0
4,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,87.0
5,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,90.0
6,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,91.0
7,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,112.0
8,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,138.0
9,NaN,NaN,NaN,1.0,Profile.entity.ein,Profile,entity.ein,,39-3842347,2.0,...,Profile,F1065.tax_year,,None,37.0,Profile.F1065.chk,Profile,F1065.chk,,141.0


In [ ]:
tx = stmtBS_Tax(llc)
try:
    ns = tx.nSpaceMap()
except Exception as err:
    print("stmtBS_Tax.nSpaceMap() {err}")
import pandas as pd
#pd.DataFrame(ns)

ValueError: All arrays must be of the same length

In [6]:
cond_shape = isinstance(ns, dict) and all(
    isinstance(v, list) and all(
        isinstance(e, dict) and {'fid', 'acct', 'fval'}.issubset(e)
        for e in v
    ) for v in ns.values()
)
flat = tx.load()
cond_flat = isinstance(flat, list) and all(
    {'fid', 'acct', 'fval', 'formNm'}.issubset(e) for e in flat
)
# Sanity: at least one form (Form1065) should have entries.
has_form1065 = bool(ns.get('Form1065'))
return _ok("stmtBS_Tax.nSpaceMap shape + flat load()",
           cond_shape and cond_flat and has_form1065,
           f"forms={list(ns.keys())} Form1065_entries="
           f"{len(ns.get('Form1065', []))}")

NameError: name 'ns' is not defined